In [23]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.3.5"
!pip install pycaret==3.3.2

  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-3.3.5.tar.gz (1.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl (28.9 MB)
Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl (9.5 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for lightgbm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [104 lines of output]
      /private/var/folders/br/qhn11dt91tvcg9cqgczhjdlr0000gn/T/pip-build-env-fzaq4u67/overlay/lib/python3.9/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider remo

In [24]:
import pandas as pd
from pycaret.regression import *
from sklearn.model_selection import train_test_split

In [25]:
SEED = 128
df = pd.read_csv("../Dataset2_Demand/6_Elec_Demand_Final.csv")

In [26]:
df["settlement_date"] = pd.to_datetime(df["settlement_date"])
df["month"] = df["settlement_date"].dt.month
df["day_of_week"] = df["settlement_date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

# Capacity utilization ratios
df["wind_utilization"] = df["embedded_wind_generation"] / (df["embedded_wind_capacity"] + 1)
df["solar_utilization"] = df["embedded_solar_generation"] / (df["embedded_solar_capacity"] + 1)

# Net cross-border flow sum
flow_cols = ["ifa2_flow","britned_flow","moyle_flow","east_west_flow","nemo_flow"]
df["net_crossborder_flow"] = df[flow_cols].sum(axis=1)

In [27]:
df_sample = df.sample(50000, random_state=SEED)  # 50k rijen

train_df, val_df = train_test_split(
    df_sample, 
    test_size=0.3, 
    random_state=SEED, 
)

reg = setup(
    data=train_df,
    target="england_wales_demand",
    session_id=SEED,
    fold=2,
    verbose=True,
)

best_model = compare_models(sort="MAE", n_select=1, turbo=True)

tuned_model = tune_model(
    best_model, 
    optimize="MAE", 
    fold=5,
    n_iter=20
)

final_model = finalize_model(tuned_model)

predictions = predict_model(final_model, data=val_df)

save_model(final_model, "../Model_ElecDemand/england_wales_demand_predictor")

,Description,Value
0,Session id,128
1,Target,england_wales_demand
2,Target type,Regression
3,Original data shape,"(35000, 21)"
4,Transformed data shape,"(35000, 23)"
5,Transformed train set shape,"(24500, 23)"
6,Transformed test set shape,"(10500, 23)"
7,Numeric features,19
8,Date features,1
9,Preprocess,True


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,985.4231,1837078.2465,1355.2177,0.9647,0.0418,0.0307,0.8600
et,Extra Trees Regressor,1028.6087,2187261.8213,1478.5487,0.9579,0.0463,0.0325,0.6250
rf,Random Forest Regressor,1098.9946,2499098.4365,1580.8243,0.9520,0.0496,0.0348,0.6100
gbr,Gradient Boosting Regressor,1411.7580,3615586.1553,1901.4286,0.9305,0.0578,0.0439,0.7600
dt,Decision Tree Regressor,1547.0754,5088712.9800,2255.7712,0.9022,0.0702,0.0487,0.0500
ada,AdaBoost Regressor,2657.5142,10494872.2691,3239.4816,0.7982,0.1067,0.0882,0.3100
knn,K Neighbors Regressor,2939.8228,14973900.8367,3869.4210,0.7121,0.1206,0.0934,0.1800
lr,Linear Regression,3673.9777,21344858.3234,4619.9685,0.5896,0.1465,0.1178,0.7450
ridge,Ridge Regression,3684.2947,21613604.8612,4648.9681,0.5845,0.1472,0.1180,0.4200
lasso,Lasso Regression,3684.4477,21613991.3027,4649.0102,0.5844,0.1472,0.1180,0.5000


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,732.7035,1039001.5342,1019.3142,0.9803,0.0311,0.0226
1,722.4752,984670.2945,992.3055,0.9809,0.0307,0.0225
2,730.7719,1008956.0356,1004.4680,0.9804,0.0316,0.0230
3,712.8233,966321.8022,983.0167,0.9815,0.0308,0.0223
4,749.4877,1083189.0337,1040.7637,0.9792,0.0322,0.0234
Mean,729.6523,1016427.7400,1007.9736,0.9805,0.0313,0.0228
Std,12.1551,41331.8599,20.4178,0.0007,0.0006,0.0004


Fitting 5 folds for each of 20 candidates, totalling 100 fits


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,649.4058,801125.8006,895.0563,0.9849,0.0280,0.0204


[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('date_feature_extractor',
                  TransformerWrapper(include=['settlement_date'],
                                     transformer=ExtractDateTimeFeatures())),
                 ('numerical_imputer',
                  TransformerWrapper(include=['settlement_period',
                                              'embedded_wind_generation',
                                              'embedded_wind_capacity',
                                              'embedded_solar_generation',
                                              'embedded_solar_capacity',
                                              'non_bm_stor',
                                              'pu...
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=[],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
        

In [28]:
from pycaret.classification import load_model
import pandas as pd

# Load model
model = load_model("../Model_ElecDemand/england_wales_demand_predictor")

# Extract features the model expects
expected_features = model[0].feature_names_in_

print("Model expects", len(expected_features), "features:")
for col in expected_features:
    print(col)

df_sample.columns


Transformation Pipeline and Model Successfully Loaded
Model expects 21 features:
settlement_date
settlement_period
embedded_wind_generation
embedded_wind_capacity
embedded_solar_generation
embedded_solar_capacity
non_bm_stor
pump_storage_pumping
ifa2_flow
britned_flow
moyle_flow
east_west_flow
nemo_flow
year
month
day_of_week
is_weekend
wind_utilization
solar_utilization
net_crossborder_flow
england_wales_demand


Index(['settlement_date', 'settlement_period', 'england_wales_demand',
       'embedded_wind_generation', 'embedded_wind_capacity',
       'embedded_solar_generation', 'embedded_solar_capacity', 'non_bm_stor',
       'pump_storage_pumping', 'ifa2_flow', 'britned_flow', 'moyle_flow',
       'east_west_flow', 'nemo_flow', 'year', 'month', 'day_of_week',
       'is_weekend', 'wind_utilization', 'solar_utilization',
       'net_crossborder_flow'],
      dtype='object')